The idea of this notebook is to selectively run the raw data from each source and process them into trajectories

# Controls

In [3]:
Mobile_Proccessing = True
WikiLoc_Proccessing = True

In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import csv
import cartopy.crs as ccrs
from shapely.geometry import Point
from shapely.geometry import LineString
import shapely.wkt
import movingpandas as mpd
from shapely.geometry import Point
from datetime import timedelta
import trackintel as ti
import glob


c:\Users\maxwe\anaconda3\envs\geospatial_study\Lib\site-packages\movingpandas\__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


# Static Vairables

In [5]:
MASK = gpd.read_file("Data/Banks_Mask_Buffered.gpkg").to_crs(epsg=2193)
SALT = 741

# Functions

In [ ]:
def clean_and_mask_gps_points(geometry, user_id ,time = None, activity_id = None, elevation = None):
    geometry = geometry.to_crs(epsg=2193)

    if time is not None:
        if pd.api.types.is_numeric_dtype(time):
            time = pd.to_datetime(time, unit='s', errors='coerce')
        else:
            time = pd.to_datetime(time, errors='coerce')

        # Keep timestamps timezone-naive for downstream movingpandas usage.
        if hasattr(time.dt, "tz") and time.dt.tz is not None:
            time = time.dt.tz_localize(None)
    
    if user_id is not None:
        user_id = str(user_id)

    if activity_id is not None:
        activity_id = str(activity_id)


    point_gdf = gpd.GeoDataFrame(
                                        geometry=geometry,  
                                        data={
                                            'time': time,
                                            'user_id': user_id,
                                            'activity_id': activity_id,
                                            'elevation': elevation
                                        })

    #Mask the GPS points using the provided mask
    masked_point_gdf = gpd.sjoin(point_gdf, MASK, how='inner', predicate='within')
    masked_point_gdf = masked_point_gdf.drop(columns=["index_right"])
    #Clean
    masked_point_gdf = masked_point_gdf.dropna(subset=["geometry", "time"])
   
    masked_point_gdf = masked_point_gdf.drop_duplicates(subset=["geometry", "time"])

    return masked_point_gdf

In [6]:
def form_trajectory(gps_point_gdf, source, activity_type=None, min_points=4, outlier_vmax=12):
    columns = [
        "user_id",
        "activity_id",
        "start_time",
        "end_time",
        "activity_type",
        "activity_class",
        "source",
        "n_points",
        "trajectory",
        "geometry"
    ]

    if gps_point_gdf is None or gps_point_gdf.empty:
        return pd.DataFrame(columns=columns)

    gps_point_gdf = gps_point_gdf.sort_values("time").copy()
    records = []

    def trajectory_geometry(traj):
        return traj.to_linestring()

    activity_type_defined = pd.notna(activity_type) and str(activity_type).strip() != ""

    def maybe_clean_outliers(traj):
        if len(traj.df) < 2:
            return traj
        if activity_type_defined:
            return mpd.OutlierCleaner(traj).clean(v_max=outlier_vmax)
        return traj

    # If there is a valid activity_id, split the activity into time-gap based trips.
    if gps_point_gdf["activity_id"].notnull().all():
        activity_id = gps_point_gdf["activity_id"].iloc[0]
        base_trajectory = mpd.Trajectory(
            gps_point_gdf,
            traj_id=activity_id,
            t="time",
            crs="EPSG:2193"
        )
        base_trajectory = maybe_clean_outliers(base_trajectory)
        trips = mpd.ObservationGapSplitter(base_trajectory).split(gap=timedelta(minutes=5))

        for trip in trips:
            if activity_type_defined:
                trip = maybe_clean_outliers(trip)

            if trip.df.empty or len(trip.df) < min_points:
                continue

            if "trip_id" in trip.df.columns:
                trip_activity_id = trip.df["trip_id"].iloc[0]
            else:
                trip_activity_id = trip.id

            # Classify the trip
            activity_class = classify_trip_activity(activity_type, trip.df)

            records.append({
                "user_id": trip.df["user_id"].iloc[0],
                "activity_id": trip_activity_id,
                "start_time": trip.get_start_time(),
                "end_time": trip.get_end_time(),
                "activity_type": activity_type,
                "activity_class": activity_class,
                "source": source,
                "n_points": len(trip.df),
                "trajectory": trip,
                "geometry": trajectory_geometry(trip)
            })
    else:
        unique_user_ids = gps_point_gdf["user_id"].dropna().unique()

        for base_user_id in unique_user_ids:
            user_gps_point_gdf = gps_point_gdf[gps_point_gdf["user_id"] == base_user_id].sort_values("time").copy()
            if user_gps_point_gdf.empty:
                continue

            base_trajectory = mpd.Trajectory(
                user_gps_point_gdf,
                traj_id=base_user_id,
                t="time",
                crs="EPSG:2193"
            )
            base_trajectory = maybe_clean_outliers(base_trajectory)
            trips = mpd.ObservationGapSplitter(base_trajectory).split(gap=timedelta(minutes=5))

            for trip in trips:
                if activity_type_defined:
                    trip = maybe_clean_outliers(trip)

                if trip.df.empty or len(trip.df) < min_points:
                    continue

                if "trip_id" in trip.df.columns:
                    trip_activity_id = trip.df["trip_id"].iloc[0]
                else:
                    trip_activity_id = trip.id

                # Classify the trip
                activity_class = classify_trip_activity(activity_type, trip.df)

                records.append({
                    "user_id": trip.df["user_id"].iloc[0],
                    "activity_id": trip_activity_id,
                    "start_time": trip.get_start_time(),
                    "end_time": trip.get_end_time(),
                    "activity_type": activity_type,
                    "activity_class": activity_class,
                    "source": source,
                    "n_points": len(trip.df),
                    "trajectory": trip,
                    "geometry": trajectory_geometry(trip)
                })

    return pd.DataFrame(records, columns=columns)

In [7]:
def classify_trip_activity(activity_type, trajectory_data=None, counts_row=None):
    """
    Classify a trip into one of three activity classes: 'pedestrian', 'cycle', or 'other'.

    Priority order:
    1. If `activity_type` is provided, classify from the activity label only (no fallback).
    2. If not provided and `counts_row` is provided, classify from aggregated counts in that row.
    3. If not provided and `trajectory_data` is provided, use mobiML-style speed heuristics.

    Parameters:
    -----------
    activity_type : str or None
        The activity type label (e.g., from Wikiloc, AllTrails)
    trajectory_data : GeoDataFrame or None
        GPS point data for mobiML-based classification if needed
    counts_row : pandas.Series or dict or None
        Optional row containing activity-count columns (e.g. activity_walking_count)

    Returns:
    --------
    str : One of 'pedestrian', 'cycle', or 'other'
    """

    # Mapping of known activity types to classes
    activity_type_mapping = {
        # Pedestrian activities
        'walking': 'pedestrian',
        'hike': 'pedestrian',
        'hiking': 'pedestrian',
        'trail': 'pedestrian',
        'trekking': 'pedestrian',
        'walk': 'pedestrian',
        'scramble': 'pedestrian',
        'running': 'pedestrian',
        'trail_running': 'pedestrian',
        'fell_running': 'pedestrian',
        'fell-running': 'pedestrian',
        'parkrun': 'pedestrian',

        # Cycling activities
        'cycling': 'cycle',
        'cycle': 'cycle',
        'bike': 'cycle',
        'biking': 'cycle',
        'mountain_biking': 'cycle',
        'mountain-biking': 'cycle',
        'mtb': 'cycle',
        'downhill': 'cycle',
        'road_biking': 'cycle',
        'road-biking': 'cycle',
        'bmx': 'cycle',
        'gravel': 'cycle',
    }

    # Check if activity_type is defined
    activity_type_defined = pd.notna(activity_type)

    if activity_type_defined:
        activity_lower = activity_type.strip().lower()

        # Direct match
        if activity_lower in activity_type_mapping:
            return activity_type_mapping[activity_lower]

        # Partial matches
        for key, classification in activity_type_mapping.items():
            if key in activity_lower:
                return classification

        # If activity_type is provided but no match found, return 'other' (no fallback)
        return 'other'


    # Fallback: Use mobiML-based classification if trajectory data available
    if trajectory_data is not None and len(trajectory_data) > 0:
        return classify_trip_by_mobiml(trajectory_data)

    # Default fallback
    return 'other'


def classify_trip_by_mobiml(trajectory_data):
    """
    Classify a trip using mobiML-based features (speed, acceleration, etc.).

    Parameters:
    -----------
    trajectory_data : GeoDataFrame
        GPS point data with geometry and time columns

    Returns:
    --------
    str : One of 'pedestrian', 'cycle', or 'other'
    """
    try:
        # Calculate statistics from trajectory
        if len(trajectory_data) < 2:
            return 'other'

        # Calculate speeds between consecutive points
        trajectory_data = trajectory_data.sort_values('time').copy()
        trajectory_data['time_diff'] = trajectory_data['time'].diff().dt.total_seconds()
        trajectory_data['distance'] = trajectory_data.geometry.distance(trajectory_data.geometry.shift())

        # Filter out invalid data
        valid_data = trajectory_data[
            (trajectory_data['time_diff'] > 0) & 
            (trajectory_data['distance'] >= 0)
        ].copy()

        if len(valid_data) < 2:
            return 'other'

        # Calculate speed (m/s)
        valid_data['speed'] = valid_data['distance'] / valid_data['time_diff']

        # Use mean speed as primary classifier
        mean_speed = valid_data['speed'].mean()

        # Heuristic classification based on speed
        # Pedestrian: typically < 3 m/s (~10 km/h)
        # Cycle: typically 3-10 m/s (~10-36 km/h)
        # Other: > 10 m/s or inconsistent
        if mean_speed < 3:  # < 10 km/h -> pedestrian
            return 'pedestrian'
        elif 3 <= mean_speed < 10:  # 10-36 km/h -> likely cycle
            return 'cycle'
        else:
            return 'other'

    except Exception as e:
        print(f"Warning: mobiML classification failed: {e}")
        return 'other'




# Mobile

## Cleaning

In [9]:
for i in range(34):
    print(f"Processing file {i+1}/34")

    gps_points = pd.read_csv(f"Data/Mobile/Download/1_{i+1}.tsv", sep="\t")
    
    geometry = gpd.points_from_xy(gps_points["Longitude of Visit"], gps_points["Latitude of Visit"],crs="EPSG:4326")
    geometry = geometry.to_crs(epsg=2193)
    user_id = gps_points["Hashed Device ID"]
    time = gps_points["Unix Timestamp of Visit"]

    masked_gps_points = clean_and_mask_gps_points(geometry, user_id, time)
    masked_gps_points.to_file(f"Data/Mobile/Cleaned/{i+1}.gpkg", driver="GPKG")


Processing file 1/34
  [LOAD] 1000057 points loaded in 1.30s
  [GEOM] Geometry created in 1.01s
    [STEP] CRS conversion: 0.001s
    [STEP] Time conversion: 0.053s
    [STEP] GeoDataFrame creation: 0.032s (1000057 points)
    [STEP] Spatial join: 0.801s (148376 points remaining)
    [STEP] Drop NA: 0.013s (148376 points)
    [STEP] Drop duplicates: 0.244s (148376 points final)
    [FUNCTION TOTAL] 1.152s
  [MASK] Points after masking: 148376 (was 1000057) in 1.19s
  [SAVE] Saved in 4.55s
  [TOTAL] File 1 completed in 8.05s

Processing file 2/34
  [LOAD] 1000001 points loaded in 1.67s
  [GEOM] Geometry created in 1.19s
    [STEP] CRS conversion: 0.001s
    [STEP] Time conversion: 0.059s
    [STEP] GeoDataFrame creation: 0.041s (1000001 points)
    [STEP] Spatial join: 0.827s (152861 points remaining)
    [STEP] Drop NA: 0.012s (152861 points)
    [STEP] Drop duplicates: 0.208s (152861 points final)
    [FUNCTION TOTAL] 1.155s
  [MASK] Points after masking: 152861 (was 1000001) in 1.21

## Trajectory Formation and Cleaning

In [ ]:
trajectory_frames = []
for i in range(34):
    print(f"Forming trajectories for file {i+1}/34")
    masked_gps_points = gpd.read_file(f"Data/Mobile/Cleaned/{i+1}.gpkg").to_crs(epsg=2193)
    trajectories = form_trajectory(masked_gps_points, source="Mobile", activity_type=None)
    if not trajectories.empty:
        trajectory_frames.append(trajectories)

trajectory_df = pd.concat(trajectory_frames, ignore_index=True)


trajectory_gdf = gpd.GeoDataFrame(trajectory_df.drop(columns=["trajectory"]), geometry="geometry", crs="EPSG:2193")
trajectory_gdf.to_file("Data/Mobile/Trajectories/trajectories.gpkg", driver="GPKG")

In [ ]:
masked_gps_points = gpd.read_file(f"Data/Mobile/Cleaned/2.gpkg").to_crs(epsg=2193)
masked_gps_points

,time,user_id,activity_id,elevation,geometry


# WikiLoc

## Cleaning

In [ ]:
activity_index = pd.read_csv("Data/Wikiloc/Downloads/activity_index.csv")
for i, activity_id in enumerate(activity_index["activity_id"], start=1):
    if i%20 == 1:
        print(f"Processing activity {i} of {len(activity_index)}")

    user_name = activity_index["user_name"][i-1]
    user_id = hash(user_name + str(SALT)) % (10 ** 8)  # Simple hash function to create a user_id from the user_name

    activity_gdf = gpd.read_file(f"Data/Wikiloc/Downloads/{activity_id}.gpx", 
                                    driver="GPX",
                                    layer="track_points", 
                                    columns=["ele", "time", "geometry"]
                                    ).to_crs(epsg=2193)

    cleaned_masked_gdf = clean_and_mask_gps_points(activity_gdf.geometry, user_id, activity_gdf.time, activity_id, activity_gdf.ele)
    cleaned_masked_gdf.to_file(f"Data/Wikiloc/Cleaned/{activity_id}.gpkg", driver="GPKG")

## Trajectory Formation and Classification

In [ ]:
activity_index = pd.read_csv("Data/Wikiloc/Downloads/activity_index.csv")
trajectory_frames = []

for i, activity_id in enumerate(activity_index["activity_id"], start=1):
    if i % 20 == 1:
        print(f"Processing activity {i} of {len(activity_index)}")

    activity_type = activity_index[activity_index["activity_id"] == activity_id]["activity_type"].values[0]

    activity_gdf = gpd.read_file(f"Data/Wikiloc/Cleaned/{activity_id}.gpkg")
    trajectories = form_trajectory(activity_gdf, source="Wikiloc", activity_type=activity_type, min_points=4)

    if not trajectories.empty:
        trajectory_frames.append(trajectories)


trajectory_df = pd.concat(trajectory_frames, ignore_index=True)


trajectory_gdf = gpd.GeoDataFrame(trajectory_df.drop(columns=["trajectory"]), geometry="geometry", crs="EPSG:2193")
trajectory_gdf.to_file("Data/Wikiloc/Trajectories/trajectories.gpkg", driver="GPKG")

Processing activity 1 of 720
Processing activity 21 of 720
Processing activity 41 of 720
Processing activity 61 of 720
Processing activity 81 of 720
Processing activity 101 of 720
Processing activity 121 of 720
Processing activity 141 of 720
Processing activity 161 of 720
Processing activity 181 of 720
Processing activity 201 of 720
Processing activity 221 of 720
Processing activity 241 of 720
Processing activity 261 of 720
Processing activity 281 of 720
Processing activity 301 of 720
Processing activity 321 of 720
Processing activity 341 of 720
Processing activity 361 of 720
Processing activity 381 of 720
Processing activity 401 of 720
Processing activity 421 of 720
Processing activity 441 of 720
Processing activity 461 of 720
Processing activity 481 of 720
Processing activity 501 of 720
Processing activity 521 of 720
Processing activity 541 of 720
Processing activity 561 of 720
Processing activity 581 of 720
Processing activity 601 of 720
Processing activity 621 of 720
Processing act

## Matched Join

In [ ]:
matched_trips = gpd.read_file("Data/Wikiloc/Matched/trajectories_matched.gpkg").to_crs(epsg=2193)
edges = gpd.read_file("Data/Map_Matching/Graph_Filter_all/edges.shp").to_crs(epsg=2193)


def split_cpath(value):
    if isinstance(value, list):
        return [part for part in value if str(part).strip()]
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(",") if part.strip()]

matched_trips["cpath"] = matched_trips["cpath"].apply(split_cpath)
matched_trips = matched_trips.explode("cpath", ignore_index=True)
matched_trips = matched_trips.dropna(subset=["cpath"])
matched_trips["cpath"] = matched_trips["cpath"].astype(int)
matched_trips["osmid"] = edges.iloc[matched_trips["cpath"].to_numpy()]["osmid"].to_numpy()

c:\Users\maxwe\anaconda3\envs\geospatial_study\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: This version of GeoPackage user_version=0x000028A0 (10400, v1.4.0) on 'Data/Wikiloc/Matched/trajectories_matched.gpkg' may only be partially supported
  return ogr_read(
c:\Users\maxwe\anaconda3\envs\geospatial_study\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Non-conformant content for record 1 in column start_time, 2021-09-20T23:39:06, successfully parsed
  return ogr_read(


## Plotting

In [ ]:
wikiloc_activity_type_counts = activity_index["activity_type"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(wikiloc_activity_type_counts.index, wikiloc_activity_type_counts.values, color=plt.cm.plasma(np.linspace(0.25, 0.9, len(wikiloc_activity_type_counts))))

ax.set_title("Wikiloc activity type distribution")
ax.set_xlabel("Count")
ax.set_ylabel("Activity type")
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()

In [ ]:
point_frames = []

if "trajectory_df" in globals() and "trajectory" in trajectory_df.columns:
    for traj in trajectory_df["trajectory"]:
        if traj is not None and hasattr(traj, "df") and not traj.df.empty:
            point_frames.append(traj.df[["geometry"]].copy())
elif "cleaned_masked_gdf" in globals() and not cleaned_masked_gdf.empty:
    point_frames.append(cleaned_masked_gdf[["geometry"]].copy())

if not point_frames:
    raise ValueError("No point data available to plot. Run the trajectory formation or cleaning cells first.")

point_gdf = gpd.GeoDataFrame(pd.concat(point_frames, ignore_index=True), geometry="geometry", crs="EPSG:2193")
point_gdf = point_gdf[point_gdf.geometry.notna()]

x = point_gdf.geometry.x
y = point_gdf.geometry.y

fig, ax = plt.subplots(figsize=(10, 8))
hb = ax.hexbin(x, y, gridsize=120, mincnt=1, bins="log", cmap="inferno")

if "MASK" in globals() and not MASK.empty:
    MASK.boundary.plot(ax=ax, color="cyan", linewidth=0.8, alpha=0.7)

cbar = fig.colorbar(hb, ax=ax)
cbar.set_label("Point density (log count)")
ax.set_title("GPS Point Density Heatmap")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

# Alltrails 

In [ ]:
track_gdf = pd.read_csv("Data/Alltrails/Downloads/banks-peninsula_track_urls.csv")


final_track_gdf = gpd.GeoDataFrame(columns=["track_name","track_id", "geometry"], geometry="geometry", crs="EPSG:2193")

for i, trail in enumerate(track_gdf["track_name"], start=1):
    #load the trail geometry
    file_name = track_gdf["file_name"].iloc[i-1]
    trail_geometry = gpd.read_file(f"Data/Alltrails/Downloads/{file_name}", 
                                    driver="GPX",layer="track_points").to_crs(epsg=2193)
    new_track_gdf = gpd.GeoDataFrame({
        "track_name": [trail],
        "track_id": [file_name.replace(".gpx", "")],
        "geometry": [LineString(trail_geometry.geometry.tolist())]
    }, geometry="geometry", crs="EPSG:2193")
    final_track_gdf = pd.concat([final_track_gdf, new_track_gdf], ignore_index=True)

final_track_gdf.to_file("Data/Alltrails/tracks.gpkg", driver="GPKG")    

In [ ]:
# Usage for AllTrails (inline counts computation)
track_gdf = gpd.read_file("Data/Alltrails/tracks.gpkg")
activity_index = pd.read_csv("Data/Alltrails/Downloads/activity_index.csv")

activity_counts = (
    activity_index.groupby(["track_name", "activity_type"]) 
    .size()
    .unstack(fill_value=0)
    .rename(columns=lambda value: f"activity_{str(value).strip().lower().replace(' ', '_').replace('-', '_')}_count")
)

track_gdf = track_gdf.set_index("track_id").join(activity_counts, how="left").reset_index()
activity_count_columns = [column for column in track_gdf.columns if column.startswith("activity_") and column.endswith("_count")]
track_gdf[activity_count_columns] = track_gdf[activity_count_columns].fillna(0).astype(int)
track_gdf["activity_count"] = track_gdf[activity_count_columns].sum(axis=1)

# Compute counts per class (pedestrian, cycle, other) based on column name keywords
pedestrian_keywords = ['walk', 'hike', 'hiking', 'trail', 'run', 'running', 'parkrun', 'trek']
cycle_keywords = ['cycle', 'bike', 'biking', 'mtb', 'mountain', 'road', 'bmx', 'gravel']

def _sum_keyword_counts(row, keywords):
    total = 0
    for col in activity_count_columns:
        col_lower = col.lower()
        if any(k in col_lower for k in keywords):
            try:
                total += float(row[col])
            except Exception:
                continue
    return int(total)

track_gdf['activity_pedestrian_count'] = track_gdf.apply(lambda r: _sum_keyword_counts(r, pedestrian_keywords), axis=1)
track_gdf['activity_cycle_count'] = track_gdf.apply(lambda r: _sum_keyword_counts(r, cycle_keywords), axis=1)

# Other count = total minus pedestrian and cycle (clipped at 0)
track_gdf['activity_other_count'] = (track_gdf['activity_count'] - track_gdf['activity_pedestrian_count'] - track_gdf['activity_cycle_count']).clip(lower=0).astype(int)

track_gdf.to_file("Data/Alltrails/tracks_with_activity_counts.gpkg", driver="GPKG")


## Plotting

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

plot_gdf = track_gdf.copy()
plot_gdf = plot_gdf[plot_gdf["activity_count"].notna()].copy()

plot_gdf.plot(
    ax=ax,
    column="activity_count",
    cmap="viridis",
    linewidth=2.5,
    legend=True,
    vmin=0,
    legend_kwds={"label": "Total activity count", "shrink": 0.75},
    missing_kwds={"color": "lightgrey", "label": "No activities"},
)
MASK.boundary.plot(ax=ax, color="cyan", linewidth=0.8, alpha=0.7)
ax.set_title("AllTrails tracks colored by total activity count")
ax.set_axis_off()
plt.tight_layout()

In [ ]:
activity_type_counts = activity_index["activity_type"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(activity_type_counts.index, activity_type_counts.values, color=plt.cm.viridis(np.linspace(0.25, 0.9, len(activity_type_counts))))

ax.set_title("AllTrails activity type distribution")
ax.set_xlabel("Count")
ax.set_ylabel("Activity type")
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()

In [ ]:

fig, ax = plt.subplots(figsize=(12, 8))
plot_gdf.sort_values("activity_count", ascending=True, inplace=True)
ax.barh(plot_gdf["track_name"], plot_gdf["activity_count"].values, color=plt.cm.viridis(np.linspace(0.25, 0.9, len(plot_gdf))))

ax.set_title("AllTrails tracks ordered by total activity count")
ax.set_xlabel("Count")
ax.set_ylabel("Track name")
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()

# Strava